# Presentation figure: improvement by seizure type

One slide for "which pre-op seizure types go with improvement?", both outcomes. Colour means the outcome throughout (**teal Engel, orange ILAE**). One row per seizure group, the same rows in every panel, ordered by the Engel odds ratio (largest at the top):

- **A. Observed improvement, Engel:** with the type (filled) vs without it (hollow), Wilson 95% CIs, improved / total.
- **B. Observed improvement, ILAE:** the same.
- **C. Odds of improvement, adjusted for LGS:** Firth two-predictor models (`improved ~ type + LGS`), Engel and ILAE, with Holm p across the 10 types.

Reads the Firth results from `1_improvement/types/` (run `pixi run run-analysis` first). Drawn at slide size (10 x 4.9 in, `cc_stats.plots.SLIDE_SIZE`). Saved as `results/<version>/presentation/improvement_by_type.png`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd

from cc_stats.outcomes import OUTCOMES
from cc_stats.paths import DATASET, results_dir
from cc_stats.plots import (INK, PAIR_A, PAIR_B, forest, paired_rate_plot, pretty, right_of, save_presentation_figure,
                            setup_or_axis, slide_figure)

print(f'dataset {DATASET}')
RESULTS = results_dir('1_improvement/types')
uni = pd.read_csv(RESULTS / 'firths_results.csv').query("predictor != 'lgs'")
adj = pd.read_csv(RESULTS / 'firths_results_lgs_adjusted.csv')
COLOR = {'engel': PAIR_A, 'ilae': PAIR_B}


def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'

In [ ]:
# Rows bottom to top (both plots draw the first row at the bottom), by the Engel adjusted OR
order = adj.query("outcome == 'engel'").sort_values('type_OR').seizure_type.tolist()
labels = [pretty(t) for t in order]
MARKS = dict(ms=5.5, lw=1.6)  # slimmer than the report figures: 20 dots per panel
DY = 0.2  # each pair's distance from its row

fig = slide_figure()
rate_axes = fig.subplots(1, 3, sharey=True, width_ratios=[1, 1, 1.2])
or_ax = rate_axes[2]

# A, B: observed improvement with vs without each type, one panel per outcome
for ax, letter, (outcome, O) in zip(rate_axes[:2], 'AB', OUTCOMES.items()):
    u = uni.query('outcome == @outcome').set_index('predictor').loc[order]
    paired_rate_plot(ax, labels, top=('with the type', u['improved_with'], u['n_with'], COLOR[outcome]),
                     bottom=('without it', u['improved_without'], u['n_without'], COLOR[outcome]),
                     header='Improved', fontsize=9, percent=False, bottom_hollow=True, offset=DY, **MARKS)
    ax.set_xticks([0, 0.5, 1])
    ax.set_xlabel(f"Improved ({O['improved_label']})")
    ax.set_title(f"{letter}.  {O['name']}: observed", loc='left', color=COLOR[outcome], fontweight='bold')

# C: LGS-adjusted ORs, Engel above ILAE, with Holm p in each outcome's colour
for outcome, offset in (('engel', DY), ('ilae', -DY)):
    a = adj.query('outcome == @outcome').set_index('seizure_type').loc[order].reset_index()
    forest(or_ax, a, ['type_OR', 'type_lo', 'type_hi'], offset=offset, color=COLOR[outcome],
           label=OUTCOMES[outcome]['name'], **MARKS)
setup_or_axis(or_ax, labels, '', ticks=(0.01, 0.1, 1, 10, 100))
or_ax.set_ylim(-0.6, len(order) - 0.05)  # headroom for the column headers (y shared)
or_ax.set_title('C.  LGS-adjusted odds', loc='left', fontweight='bold')
or_ax.text(1, len(order) - 0.35, 'Holm p', transform=right_of(or_ax, 0.35), va='center', ha='center',
           fontsize=9, fontweight='bold')
for outcome, offset in (('engel', DY), ('ilae', -DY)):
    a = adj.query('outcome == @outcome').set_index('seizure_type').loc[order]
    for k, v in enumerate(a['type_p_holm']):
        or_ax.text(1, k + offset, fmt_p(v), transform=right_of(or_ax, 0.35), va='center', ha='center',
                   fontsize=9, color=COLOR[outcome])

# One legend for the slide, in ink: colour already says the outcome (panel titles), so the legend only
# explains filled vs hollow (A, B) and which line is which (C)
legend = [Line2D([], [], marker='o', ls='', ms=6, color=INK, label='with the type'),
          Line2D([], [], marker='o', ls='', ms=6, color=INK, mfc='white', mew=1.6, label='without it'),
          Line2D([], [], marker='o', ms=6, lw=1.6, color=PAIR_A, label='Engel'),
          Line2D([], [], marker='o', ms=6, lw=1.6, color=PAIR_B, label='ILAE')]
fig.legend(handles=legend, loc='outside lower center', ncol=4, frameon=False, fontsize=9.5, columnspacing=2.5)

fig.get_layout_engine().set(rect=(0, 0, 0.965, 1), w_pad=0.12)  # room for the Holm p column; air between panels
save_presentation_figure('improvement_by_type', fig)
plt.show()